# Notebook 03 — KNN 



## Step 0 — Setup

In [1]:
import os, warnings, pickle, copy, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings("ignore")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results


## Step 1 — Load Cached Features and Train/Val/Test Split

**Fix (critical)**: same hard assertion as the corrected RF notebook --
this is exactly what would have caught the stale-cache reuse here too.


In [2]:
HC_CACHE    = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")
DEEP_CACHE  = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

X_hc = np.load(HC_CACHE, allow_pickle=True)["X"]
print(f"Handcrafted features: {X_hc.shape}")

if os.path.exists(DEEP_CACHE):
    X_deep = np.load(DEEP_CACHE, allow_pickle=True)["X"]
    X_all  = np.concatenate([X_deep, X_hc], axis=1)
    print(f"Deep features: {X_deep.shape}")
else:
    X_all = X_hc
    print("Deep features not found -- using handcrafted only")
print(f"Fused features: {X_all.shape}")

split = np.load(SPLIT_CACHE, allow_pickle=True)
X_train_sc = split["X_train_sc"]
X_val_sc   = split["X_val_sc"]
X_test_sc  = split["X_test_sc"]
y_train_sm = split["y_train_sm"]
y_val_raw  = split["y_val_raw"]
y_test_raw = split["y_test_raw"]
n_components = int(split["n_components"])

with open(os.path.join(OUTPUT_DIR, "label_encoder_species.pkl"), "rb") as f:
    le = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc1.pkl"), "rb") as f:
    sc1 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc2.pkl"), "rb") as f:
    sc2 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "pca.pkl"), "rb") as f:
    pca = pickle.load(f)

class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {NUM_CLASSES} -- {class_names}")



Handcrafted features: (19145, 73)
Deep features: (19145, 1536)
Fused features: (19145, 1609)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]


## Step 2 — Evaluation Utilities (identical to SVM/RF)

In [3]:
N_BOOTSTRAP = 1000
SMALL_SUPPORT_THRESHOLD = 30

def evaluate(clf, Xtr, Xte, ytr, yte):
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    yproba = None
    auc_s = None
    if hasattr(clf, "predict_proba"):
        try:
            yproba = clf.predict_proba(Xte)
            auc_s = round(roc_auc_score(yte, yproba, multi_class="ovr",
                                         average="macro") * 100, 2)
        except Exception:
            pass
    return {
        "Accuracy":  round(accuracy_score(yte, yp) * 100, 2),
        "Precision": round(precision_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "Recall":    round(recall_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "F1":        round(f1_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "AUC": auc_s,
        "_yp": yp, "_yproba": yproba, "_fitted_clf": clf,
    }

def bootstrap_ci(y_true, y_pred, y_proba=None, n_boot=N_BOOTSTRAP, seed=GLOBAL_SEED):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        precs.append(precision_score(yt, yp, average="weighted", zero_division=0))
        recs.append(recall_score(yt, yp, average="weighted", zero_division=0))
        f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        if y_proba is not None:
            try:
                aucs.append(roc_auc_score(yt, y_proba[idx], multi_class="ovr", average="macro"))
            except ValueError:
                pass

    def summarize(arr):
        arr = np.array(arr) * 100
        return {"mean": round(float(arr.mean()), 2),
                "ci_lo": round(float(np.percentile(arr, 2.5)), 2),
                "ci_hi": round(float(np.percentile(arr, 97.5)), 2)}

    result = {"Accuracy": summarize(accs), "Precision": summarize(precs),
              "Recall": summarize(recs), "F1": summarize(f1s)}
    if y_proba is not None:
        result["AUC"] = summarize(aucs) if aucs else None
        result["AUC_valid_resamples"] = len(aucs)
        result["AUC_skipped_resamples"] = n_boot - len(aucs)
    return result

def per_class_report(y_true, y_pred, class_names, small_support_threshold=SMALL_SUPPORT_THRESHOLD):
    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for cls in class_names:
        r = report[cls]
        rows.append({"Class": cls, "Precision": round(r["precision"] * 100, 2),
                     "Recall": round(r["recall"] * 100, 2),
                     "F1": round(r["f1-score"] * 100, 2), "Support": int(r["support"])})
    df = pd.DataFrame(rows)
    small = df[df["Support"] < small_support_threshold]
    if not small.empty:
        print(f"  \u26a0 Small test support (<{small_support_threshold} samples) -- "
              f"per-class scores below should not be over-interpreted (comment 7):")
        for _, r in small.iterrows():
            print(f"     {r['Class']:<28} support={r['Support']:>4}  "
                  f"P={r['Precision']:.1f}%  R={r['Recall']:.1f}%  F1={r['F1']:.1f}%")
    return df

def save_full_result(name, m, y_true, class_names):
    clf = m["_fitted_clf"]
    yp, yproba = m["_yp"], m["_yproba"]

    ci = bootstrap_ci(y_true, yp, yproba)
    per_class = per_class_report(y_true, yp, class_names)
    cm = confusion_matrix(y_true, yp, labels=list(range(len(class_names))))

    result = {k: v for k, v in m.items() if not k.startswith("_")}
    result["Classifier"] = name
    result["Bootstrap_CI_95"] = ci
    result["N_bootstrap"] = N_BOOTSTRAP
    result["PerClass"] = per_class.to_dict(orient="records")

    out_path = os.path.join(OUTPUT_DIR, f"result_{name.replace(' ','_')}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    mdl_path = os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl")
    with open(mdl_path, "wb") as f:
        pickle.dump(clf, f)

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.replace(' ','_')}.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    auc_s = f"{m['AUC']}%" if m["AUC"] is not None else "n/a"
    print(f"  Acc={m['Accuracy']:6.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"P={m['Precision']:6.2f}%  R={m['Recall']:6.2f}%  F1={m['F1']:6.2f}% "
          f"[{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={auc_s}")
    print(f"  Saved -> {out_path}")
    print(f"  Saved -> {mdl_path}")
    print(f"  Saved -> {cm_path}")

    with open(mdl_path, "rb") as f:
        reloaded = pickle.load(f)
    _ = reloaded.predict(X_test_sc[:1])
    print(f"  Smoke-test passed: reloaded model predicts without error.")

    return result

print("Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).")


Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).


## Step 3 — Baseline KNN (k=5, matches original notebook)

In [4]:
from sklearn.neighbors import KNeighborsClassifier

name = "KNN"
print(f"Training {name} ...")
print("AUC methodology: KNeighborsClassifier's native predict_proba "
      "(neighbor-vote-fraction based) used directly -- same reasoning as "
      "Random Forest: ROC-AUC is rank-based and does not require calibrated "
      "probabilities. AUC is One-vs-Rest, macro-averaged.")

clf = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

m = evaluate(clf, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
result_baseline = save_full_result(name, m, y_test_raw, class_names)


Training KNN ...
AUC methodology: KNeighborsClassifier's native predict_proba (neighbor-vote-fraction based) used directly -- same reasoning as Random Forest: ROC-AUC is rank-based and does not require calibrated probabilities. AUC is One-vs-Rest, macro-averaged.
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 77.13% [75.5-78.8]  P= 83.81%  R= 77.13%  F1= 79.00% [77.5-80.5]  AUC=96.14%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_KNN.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_KNN.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_KNN.csv
  Smoke-test passed: reloaded model predicts without error.


## Step 4 — Optuna Hyperparameter Search (new; comment 10)

Baseline accuracy (78.4% in the original run) is notably lower than SVM/RF
(90%+) -- k=5 with default weighting/metric was never tuned for this
300-dim, SMOTE-balanced feature space. Tuned on the held-out **validation**
split; Test is not touched here.


In [5]:
import optuna
from optuna.samplers import TPESampler

N_TRIALS = 40
SEARCH_SPACE_DESC = (
    "n_neighbors ~ Int[1,50], weights in {'uniform','distance'}, "
    "metric in {'euclidean','manhattan','cosine'}"
)
print(f"Optuna search space (comment 10): {SEARCH_SPACE_DESC}")
print(f"seed={GLOBAL_SEED}, n_trials={N_TRIALS}, sampler=TPESampler")
print("Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.")

def objective_knn(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    weights     = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric      = trial.suggest_categorical("metric", ["euclidean", "manhattan", "cosine"])
    # cosine distance is not supported by the default KD-tree/ball-tree
    # algorithms in scikit-learn -- brute force is required for it.
    algorithm = "brute" if metric == "cosine" else "auto"
    clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights,
                                metric=metric, algorithm=algorithm, n_jobs=-1)
    clf.fit(X_train_sc, y_train_sm)
    yp = clf.predict(X_val_sc)
    return f1_score(y_val_raw, yp, average="weighted")

study_knn = optuna.create_study(direction="maximize", sampler=TPESampler(seed=GLOBAL_SEED))
study_knn.optimize(objective_knn, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest VAL F1(weighted) = {study_knn.best_value*100:.2f}%")
print(f"Best params: {study_knn.best_params}")

optuna_best_knn = {
    "Classifier": "KNN_Optuna_BestTrial",
    "Val_F1_weighted": round(study_knn.best_value * 100, 2),
    "BestParams": study_knn.best_params,
    "NumTrials": N_TRIALS,
    "SearchSpace": SEARCH_SPACE_DESC,
    "Sampler": "TPESampler",
    "Seed": GLOBAL_SEED,
    "TuningSet": "validation (held out, not SMOTE-resampled)",
}
with open(os.path.join(OUTPUT_DIR, "optuna_result_KNN.json"), "w") as f:
    json.dump(optuna_best_knn, f, indent=2)
print(optuna_best_knn)


[I 2026-10-19 14:33:57,685] A new study created in memory with name: no-name-c0261479-60a7-4724-870f-845b97a57871


Optuna search space (comment 10): n_neighbors ~ Int[1,50], weights in {'uniform','distance'}, metric in {'euclidean','manhattan','cosine'}
seed=42, n_trials=40, sampler=TPESampler
Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-10-19 14:33:58,031] Trial 0 finished with value: 0.7674347181511534 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.7674347181511534.
[I 2026-10-19 14:33:59,806] Trial 1 finished with value: 0.8536694390028567 and parameters: {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 1 with value: 0.8536694390028567.
[I 2026-10-19 14:34:01,511] Trial 2 finished with value: 0.8764403956814121 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'metric': 'cosine'}. Best is trial 2 with value: 0.8764403956814121.
[I 2026-10-19 14:34:03,086] Trial 3 finished with value: 0.878216048694796 and parameters: {'n_neighbors': 22, 'weights': 'distance', 'metric': 'cosine'}. Best is trial 3 with value: 0.878216048694796.
[I 2026-10-19 14:34:07,472] Trial 4 finished with value: 0.7762657234080795 and parameters: {'n_neighbors': 23, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 3 with value: 0

## Step 5 — Final Model: Retrain with Best Hyperparameters, Evaluate Once on Test

In [6]:
name = "KNN_Optuna"
print(f"Training {name} (final retrained model, evaluated ONCE on held-out test) ...")

bp = study_knn.best_params
algorithm_final = "brute" if bp["metric"] == "cosine" else "auto"
clf_final = KNeighborsClassifier(n_neighbors=bp["n_neighbors"], weights=bp["weights"],
                                  metric=bp["metric"], algorithm=algorithm_final, n_jobs=-1)

m_final = evaluate(clf_final, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
result_optuna = save_full_result(name, m_final, y_test_raw, class_names)

print("\n" + "="*70)
print("BASELINE (k=5, default) vs. OPTUNA-TUNED -- compare on TEST:")
print("="*70)
print(f"  Baseline : Acc={result_baseline['Accuracy']:.2f}%  F1={result_baseline['F1']:.2f}%  "
      f"AUC={result_baseline['AUC']}%")
print(f"  Optuna   : Acc={result_optuna['Accuracy']:.2f}%  F1={result_optuna['F1']:.2f}%  "
      f"AUC={result_optuna['AUC']}%")


Training KNN_Optuna (final retrained model, evaluated ONCE on held-out test) ...
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 87.66% [86.3-89.0]  P= 89.30%  R= 87.66%  F1= 88.08% [86.9-89.4]  AUC=98.63%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_KNN_Optuna.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_KNN_Optuna.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_KNN_Optuna.csv
  Smoke-test passed: reloaded model predicts without error.

BASELINE (k=5, default) vs. OPTUNA-TUNED -- compare on TEST:
  Baseline : Acc=77.13%  F1=79.00%  AUC=96.14%
  Optuna   : Acc=87.66%  F1=88.08%  AUC=98.63%
